# ECE 685D — Project 1 Runbook
**Hypernetworks for Implicit Neural Representations**

Run cells top-to-bottom. Each section is self-contained.

| Section | Content |
|---------|--------|
| 0 | Setup & GPU check |
| 1 | CIFAR-10 reconstruction — SIREN / Fourier / ReLU |
| 2 | CIFAR-10 downstream classification — latent / theta / pixels |
| 3 | CIFAR-10 figure export & visual comparison |
| 4 | CelebA reconstruction — SIREN / Fourier / ReLU |
| 5 | Super-resolution — all three architectures (64px→256px) |
| 6 | CelebA & super-res figure export |
| 7 | Full results summary — tables, efficiency, training curves |

## Section 0 — Setup

In [ ]:
import os
os.chdir('/home/zihan-gao/dl685/project/project1')

In [ ]:
!pip install lpips -q

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU: ', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

---
## Section 1 — CIFAR-10 Reconstruction

Train hypernetwork H(I)→θ for three coordinate-network architectures on CIFAR-10 (32×32).
Results saved to `runs/cifar10_<arch>_recon.pt` and `runs/cifar10_<arch>_recon_metrics.jsonl`.

In [ ]:
# SIREN — sinusoidal activations, per-layer weight scaling (~30 min)
!python train_reconstruction.py \
  --dataset cifar10 --arch siren --image-size 32 \
  --epochs 120 --batch-size 128 --samples-per-image 4096 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --lr 3e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 50 --out-dir ./runs

In [ ]:
# Fourier — random Gaussian frequency features (~30 min)
!python train_reconstruction.py \
  --dataset cifar10 --arch fourier --image-size 32 \
  --epochs 120 --batch-size 128 --samples-per-image 4096 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --fourier-features 32 --fourier-sigma 10 \
  --lr 3e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 50 --out-dir ./runs

In [ ]:
# ReLU — plain MLP baseline (~30 min)
!python train_reconstruction.py \
  --dataset cifar10 --arch relu --image-size 32 \
  --epochs 120 --batch-size 128 --samples-per-image 4096 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --lr 3e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 50 --out-dir ./runs

In [ ]:
import json, glob
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for path in sorted(glob.glob('./runs/cifar10_*_recon_metrics.jsonl')):
    arch = path.split('cifar10_')[1].split('_recon')[0]
    rows = [json.loads(l) for l in open(path)]
    for ax, m, lab in zip(axes, ['psnr', 'ssim', 'lpips'], ['PSNR (dB)↑', 'SSIM↑', 'LPIPS↓']):
        vals = [r[m] for r in rows if m in r]
        ax.plot(range(1, len(vals)+1), vals, label=arch)
        ax.set_xlabel('Epoch'); ax.set_ylabel(lab); ax.legend(); ax.grid(True)
plt.suptitle('CIFAR-10 — Validation Metrics')
plt.tight_layout()
plt.savefig('./runs/cifar10_training_curves.png', dpi=150)
plt.show()

---
## Section 2 — CIFAR-10 Downstream Classification

Linear probe on three feature types:
- **latent** — hypernetwork bottleneck z (512-dim)
- **theta** — full generated INR weights (~50k-dim)
- **pixels** — raw flattened pixels (3072-dim) — baseline


In [ ]:
!python train_downstream_classifier.py \
  --recon-ckpt ./runs/cifar10_siren_recon.pt \
  --feature-type latent --cls-epochs 100 \
  --out-path ./runs/cifar10_siren_latent_cls.pt \
  --max-train-samples 0 --max-test-samples 0


In [ ]:
!python train_downstream_classifier.py \
  --recon-ckpt ./runs/cifar10_siren_recon.pt \
  --feature-type theta --cls-epochs 100 \
  --out-path ./runs/cifar10_siren_theta_cls.pt \
  --max-train-samples 0 --max-test-samples 0


In [ ]:
# Raw pixel baseline — no checkpoint needed
!python train_downstream_classifier.py \
  --feature-type pixels --cls-epochs 100 \
  --out-path ./runs/cifar10_pixels_cls.pt \
  --max-train-samples 0 --max-test-samples 0


In [ ]:
import torch, glob
print(f'{"Feature":<35}  {"Best Acc":>10}  {"Dim":>8}')
print('-' * 58)
for path in sorted(glob.glob('./runs/cifar10_*_cls.pt')):
    c = torch.load(path, map_location='cpu')
    name = path.split('/')[-1].replace('.pt', '')
    print(f'{name:<35}  {c["best_acc"]:>9.1f}%  {c["in_dim"]:>8}')

---
## Section 3 — CIFAR-10 Figure Export

In [ ]:
import os
for arch in ['siren', 'fourier', 'relu']:
    os.system(
        f'python export_recon_examples.py'
        f' --recon-ckpt ./runs/cifar10_{arch}_recon.pt'
        f' --num-images 30'
        f' --out-dir ./runs/figures/cifar10_{arch}'
    )

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

archs = ['siren', 'fourier', 'relu']
n_rows = 5
fig, axes = plt.subplots(n_rows, len(archs) * 2, figsize=(len(archs) * 5, n_rows * 1.6))
for row in range(n_rows):
    for col, arch in enumerate(archs):
        tgt   = Image.open(f'./runs/figures/cifar10_{arch}/{row:04d}_target.png')
        recon = Image.open(f'./runs/figures/cifar10_{arch}/{row:04d}_recon.png')
        axes[row, col*2].imshow(tgt);     axes[row, col*2].axis('off')
        axes[row, col*2+1].imshow(recon); axes[row, col*2+1].axis('off')
        if row == 0:
            axes[0, col*2].set_title(f'{arch}\nTarget')
            axes[0, col*2+1].set_title(f'{arch}\nRecon')
plt.suptitle('CIFAR-10 Reconstructions — Target vs Recon')
plt.tight_layout()
plt.savefig('./runs/figures/cifar10_comparison.png', dpi=150)
plt.show()

### CelebA Data Setup

CelebA cannot be auto-downloaded from Google Drive (rate-limited for academic datasets).  
Run the cell below once to fetch it via **Kaggle**:
1. Create an account at [kaggle.com](https://www.kaggle.com)
2. Go to **Account → Settings → API → Create New Token** — downloads `kaggle.json`
3. In a terminal: `mkdir -p ~/.kaggle && cp ~/Downloads/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json`
4. Run the cell below

If CelebA is already extracted in `./data/celeba/img_align_celeba/`, the cell will just report the count and skip.

In [ ]:
import subprocess, shutil
from pathlib import Path

img_dir = Path('./data/celeba/img_align_celeba')
n_imgs = len(list(img_dir.glob('*.jpg'))) if img_dir.exists() else 0
print(f'Found {n_imgs} CelebA images in {img_dir}')

if n_imgs < 180_000:
    print('Downloading CelebA via Kaggle (~1.4 GB)...')
    subprocess.run(['pip', 'install', 'kaggle', '-q'], check=True)

    dl_dir = Path('./celeba_kaggle_dl')
    dl_dir.mkdir(exist_ok=True)
    subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', 'jessicali9530/celeba-dataset',
         '-p', str(dl_dir), '--unzip'],
        check=True
    )

    # Locate images (Kaggle dataset may be nested one level)
    img_dir.mkdir(parents=True, exist_ok=True)
    for candidate in [
        dl_dir / 'img_align_celeba' / 'img_align_celeba',
        dl_dir / 'img_align_celeba',
        dl_dir / 'celeba' / 'img_align_celeba',
    ]:
        if candidate.exists() and list(candidate.glob('*.jpg')):
            for f in candidate.glob('*.jpg'):
                shutil.move(str(f), str(img_dir / f.name))
            print(f'Moved images from {candidate}')
            break

    shutil.rmtree(dl_dir, ignore_errors=True)
    n_imgs = len(list(img_dir.glob('*.jpg')))
    print(f'Setup complete: {n_imgs} images')
else:
    print('CelebA already set up.')


---
## Section 4 — CelebA Reconstruction

Same three architectures on CelebA (128×128). Compare quality, stability, and speed.

In [ ]:
# SIREN (~60–90 min)
!python train_reconstruction.py \
  --dataset celeba --arch siren --image-size 128 \
  --epochs 60 --batch-size 32 --samples-per-image 8192 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --lr 2e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 30 --out-dir ./runs

In [ ]:
# Fourier (~60–90 min)
!python train_reconstruction.py \
  --dataset celeba --arch fourier --image-size 128 \
  --epochs 60 --batch-size 32 --samples-per-image 8192 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --fourier-features 32 --fourier-sigma 10 \
  --lr 2e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 30 --out-dir ./runs

In [ ]:
# ReLU (~60–90 min)
!python train_reconstruction.py \
  --dataset celeba --arch relu --image-size 128 \
  --epochs 60 --batch-size 32 --samples-per-image 8192 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --lr 2e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 30 --out-dir ./runs

---
## Section 5 — Super-Resolution

Protocol: train on **64×64** CelebA, query coordinate network at **256×256** (4× SR).
Checkpoints saved as `celeba_<arch>_superres64to256_recon.pt` (separate from Section 4).

In [ ]:
# SIREN super-resolution
!python train_reconstruction.py \
  --dataset celeba --arch siren --image-size 128 \
  --train-lowres 64 --superres-eval-size 256 \
  --epochs 60 --batch-size 32 --samples-per-image 8192 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --lr 2e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 30 --out-dir ./runs

In [ ]:
# Fourier super-resolution
!python train_reconstruction.py \
  --dataset celeba --arch fourier --image-size 128 \
  --train-lowres 64 --superres-eval-size 256 \
  --epochs 60 --batch-size 32 --samples-per-image 8192 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --fourier-features 32 --fourier-sigma 10 \
  --lr 2e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 30 --out-dir ./runs

In [ ]:
# ReLU super-resolution
!python train_reconstruction.py \
  --dataset celeba --arch relu --image-size 128 \
  --train-lowres 64 --superres-eval-size 256 \
  --epochs 60 --batch-size 32 --samples-per-image 8192 \
  --hidden-dim 128 --depth 4 --latent-dim 512 \
  --lr 2e-4 --weight-decay 1e-7 \
  --num-workers 8 --amp --eval-batches 30 --out-dir ./runs

---
## Section 6 — Figure Export

In [ ]:
import os
for arch in ['siren', 'fourier', 'relu']:
    os.system(
        f'python export_recon_examples.py'
        f' --recon-ckpt ./runs/celeba_{arch}_recon.pt'
        f' --num-images 20'
        f' --out-dir ./runs/figures/celeba_{arch}'
    )

In [ ]:
import os
for arch in ['siren', 'fourier', 'relu']:
    os.system(
        f'python export_recon_examples.py'
        f' --recon-ckpt ./runs/celeba_{arch}_superres64to256_recon.pt'
        f' --train-lowres 64 --output-size 256'
        f' --num-images 10'
        f' --out-dir ./runs/figures/celeba_{arch}_superres256'
    )

---
## Section 7 — Full Results Summary

In [ ]:
import json, glob

print(f'{"Model":<42}  {"Best PSNR":>9}  {"SSIM":>8}  {"LPIPS":>8}  {"Epoch":>6}')
print('-' * 80)
for path in sorted(glob.glob('./runs/*_recon_metrics.jsonl')):
    rows = [json.loads(l) for l in open(path)]
    best = max(rows, key=lambda r: r.get('psnr', -1))
    name = path.split('/')[-1].replace('_recon_metrics.jsonl', '')
    lpips = f"{best['lpips']:.4f}" if 'lpips' in best else '   N/A'
    print(f'{name:<42}  {best["psnr"]:>9.2f}  {best["ssim"]:>8.4f}  {lpips:>8}  {best["epoch"]:>6}')

In [ ]:
import torch, glob

print(f'{"Classifier":<35}  {"Best Acc":>10}  {"Feature":>10}  {"Dim":>8}')
print('-' * 68)
for path in sorted(glob.glob('./runs/*_cls.pt')):
    c = torch.load(path, map_location='cpu')
    name = path.split('/')[-1].replace('.pt', '')
    print(f'{name:<35}  {c["best_acc"]:>9.1f}%  {c["feature_type"]:>10}  {c["in_dim"]:>8}')

In [ ]:
import json, glob

print(f'{"Model":<42}  {"Avg sec/epoch":>14}  {"Epochs":>7}')
print('-' * 68)
for path in sorted(glob.glob('./runs/*_recon_metrics.jsonl')):
    rows = [json.loads(l) for l in open(path)]
    avg_t = sum(r['time_sec'] for r in rows) / len(rows)
    name = path.split('/')[-1].replace('_recon_metrics.jsonl', '')
    print(f'{name:<42}  {avg_t:>14.1f}s  {len(rows):>7}')

In [ ]:
import json, glob, matplotlib.pyplot as plt

datasets = ['cifar10', 'celeba']
metrics  = [('psnr', 'PSNR (dB)↑'), ('ssim', 'SSIM↑'), ('lpips', 'LPIPS↓')]
fig, axes = plt.subplots(len(datasets), len(metrics), figsize=(16, 9))

for row, ds in enumerate(datasets):
    for col, (metric, label) in enumerate(metrics):
        ax = axes[row][col]
        for path in sorted(glob.glob(f'./runs/{ds}_*_recon_metrics.jsonl')):
            if 'superres' in path:
                continue
            arch = path.split(f'{ds}_')[1].split('_recon')[0]
            rows = [json.loads(l) for l in open(path)]
            vals = [r[metric] for r in rows if metric in r]
            ax.plot(range(1, len(vals)+1), vals, label=arch)
        ax.set_title(f'{ds.upper()} — {label}')
        ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig('./runs/all_training_curves.png', dpi=150)
plt.show()

In [ ]:
import json, glob, matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for path in sorted(glob.glob('./runs/*_superres*_recon_metrics.jsonl')):
    arch = path.split('celeba_')[1].split('_superres')[0]
    rows = [json.loads(l) for l in open(path)]
    for ax, metric, label in zip(axes, ['psnr', 'ssim', 'lpips'], ['PSNR (dB)↑', 'SSIM↑', 'LPIPS↓']):
        vals = [r[metric] for r in rows if metric in r]
        ax.plot(range(1, len(vals)+1), vals, label=arch)
        ax.set_xlabel('Epoch'); ax.set_ylabel(label); ax.legend(); ax.grid(True)
plt.suptitle('Super-Resolution (64px→256px) — Architecture Comparison')
plt.tight_layout()
plt.savefig('./runs/superres_curves.png', dpi=150)
plt.show()